In [ ]:
zPath   = None
no_bins = None
MK      = None
size    = None
sigma_p = None
lvl_ref = None

In [1]:
zPath = '170'
no_bins = 121
MK = 'MK2'
size = 512
sigma_p = 0.6
lvl_ref = 100000

In [2]:
#%run ./0___Reference.ipynb
#%run ./0___Reference_functions.ipynb
%run ./0___Reference_Plots.ipynb
#%run ./0___Reference_Shapes.ipynb
#%run ./0___Reference_Follow_Curve.ipynb

In [3]:
#from scipy.interpolate import UnivariateSpline

In [4]:
import os

---
---
---
---
---
---
---
---
---
---

In this usecase for the void finder, we use the data from the 2D histograms.

In [7]:
if not os.path.exists(plots_path_0+"14___Shape_Hist/11.5___Follow_Curve_"+str(int(no_bins))+"/Z_"+zPath+"/hist/" ): os.makedirs(plots_path_0+"14___Shape_Hist/11.5___Follow_Curve_"+str(int(no_bins))+"/Z_"+zPath+"/hist/" )
if not os.path.exists(plots_path_0+"14___Shape_Hist/11.5___Follow_Curve_"+str(int(no_bins))+"/Z_"+zPath+"/plt/"  ): os.makedirs(plots_path_0+"14___Shape_Hist/11.5___Follow_Curve_"+str(int(no_bins))+"/Z_"+zPath+"/plt/"  )
if not os.path.exists(plots_path_0+"14___Shape_Hist/11.5___Follow_Curve_"+str(int(no_bins))+"/Z_"+zPath+"/angle/"): os.makedirs(plots_path_0+"14___Shape_Hist/11.5___Follow_Curve_"+str(int(no_bins))+"/Z_"+zPath+"/angle/")

---

In [8]:
x_label = "Distance from the void's center\nas a percentage of their radius" 
y_label = 'Density per percentage\nof radii (per void) '+r'[$\times 10^{10}$M$_\odot$(cMpc/h)$^{-3}$]'

In [ ]:
with open(plots_path_0+"11___Shape_Hist/11.5___Follow_Curve_"+str(int(no_bins))+"/Z_"+zPath+"/x.pk", 'rb') as f: x = np.array(pkl.load(f))
with open(plots_path_0+"11___Shape_Hist/11.5___Follow_Curve_"+str(int(no_bins))+"/Z_"+zPath+"/y.pk", 'rb') as f: y = np.array(pkl.load(f))

In [15]:
analysis_path = '../Analysis___Ellipsoid_Data/'
Dns_ud = 0.2
Dns_od = 0.3
zPath = "01"
file_path_Z = analysis_path+"D___["+str(Dns_ud)+"_"+str(Dns_od)+"]/Z___"+zPath+"/"

In [16]:
with open(file_path_Z+"pixels_scaledx_scaledy.pk", 'rb') as f: pixels_scaledx_scaledy = pkl.load(f)

In [ ]:
sort_indices = np.argsort(x)
x = x[sort_indices]
y = y[sort_indices]

---

## The edge region cut function

Form an upper and lower curve that will serve as a tangentual fit to our currect point to be fitted in terms of the lean angle it should adopt.

In [ ]:
unique_x = np.unique(x)

# We found the mean of the top and bottom 10 values in each bin to be a better fit for data that has some spread outside of its main curve
upper_bound = np.array([np.mean(np.partition(y[x == xi], -10)[-10:]) for xi in unique_x])
lower_bound = np.array([np.mean(np.partition(y[x == xi],  10)[:10 ]) for xi in unique_x])

#upper_bound = np.array([np.max(y[x == xi]) for xi in unique_x])
#lower_bound = np.array([np.min(y[x == xi]) for xi in unique_x])

Smooth the upper and lower curves.

In [ ]:
smoothing_factor = 5000
upper_curve = UnivariateSpline(unique_x, upper_bound, s=smoothing_factor)
lower_curve = UnivariateSpline(unique_x, lower_bound, s=smoothing_factor)

In [ ]:
x_smooth = np.linspace(0, no_bins, 6000)
upper_smooth = upper_curve(x_smooth)
lower_smooth = lower_curve(x_smooth)

upper_unique = upper_curve(unique_x)
lower_unique = lower_curve(unique_x)

In [ ]:
x_cut = []; y_cut = []
mask = np.zeros_like(x, dtype=bool)

for i, ux in enumerate(unique_x):
    indices = np.where(x == ux)
    mask[indices] = (y[indices] >= lower_unique[i]) & (y[indices] <= upper_unique[i])

x_cut  = x[ mask]; y_cut  = y[ mask]
x_cutt = x[~mask]; y_cutt = y[~mask]

We need a very large dpi, or else visual artifacts appear for grid plots such as these.

Don't be fooled!

In [ ]:
if False:
    
    plt.figure(dpi=800, figsize=(4,4))
    
    
    plt.hist2d(x_cut, y_cut, bins = [np.arange(0, no_bins, 1), np.arange(0, no_bins, 1)], cmin=0.1, norm=mpl.colors.LogNorm())
    plt.scatter(x_cutt+0.5, y_cutt+0.5, c='r', s=1,   lw=0, alpha=0.6)
    
    plt.scatter(x_smooth+0.5, upper_smooth+0.5, c='blue', s=0.5, lw=0)
    plt.scatter(x_smooth+0.5, lower_smooth+0.5, c='blue', s=0.5, lw=0)
    
    plt.ylabel(y_label)
    plt.xlabel(x_label)
    plt.tight_layout()
    plt.show()

---
---
---

## Set the variables.

Toy with:
- starting_point (just once to see if there's some random variation which might cause issues);
- minima_slit_size (multiple ones... see how they affect computational time and different results in combination to increasing:)
- minima_slit_count & maxima_slit_count - (see above as a combinaiton);
- advance_ratio - also toy with this, but independently of the rest.

We see that the code executed the starting point iteration using the minima_slit_size, but that wasn't enough to get the minima_slit_count, so it increased the former. Maybe too much, but if so, then it reduced to so that the final count is now between the minima_slit_count and the maxima_slit_count.

## The first point fit iteration.

We see that, by simply giving the elements_finder function an interval to look for the closest guess, we have shaved-off the time from 8.8s down to 0.4s. This is huge. We can now look for as many angles as we want to find the best tangent.

Also, we observe that cutting the min-max count from 900-1500 down to 900-1000 (a 6-fold cut in the interval), we don't take much longer. The algorithm is very efficient in that part. This means we can aim for very good (by means of similarity) statistics along the fit points.

Also, we see a very good indication that the bigger the angle, the more elements we will have in the list, since we start with the same slit_size as taken from the original first point fit. This is as expected, doesn't tell us much new. However, we see below that the chosen angle is dead center (the 17th iteration out of 33). This signals that, even when we look for the smallest spread in the beta_prime distribution, we do seem to get the result closest to the best inclination (since here we knew the best one is the dead-center one as we were in the flat region and our initial slope was 1000 - the limit we set for slopes, just so stuff wouldn't blow-up or take too much or whatevs).

In [ ]:
back_forward = [0,1]   # [0,1]
lr = ["left", "right"]
pm = [1, -1]

In [ ]:
# We chose half-way point, where the vertical coordinate is the average of the upper and lower bound curves there.
starting_point = [no_bins/2, (upper_unique[int(no_bins/2)] + lower_unique[int(no_bins/2)])/2]

In [ ]:
# Finds the elemetns in verticla bins. The way this works for angles is we first rotate the data, then we do vertical bins.
length_bin_0 = no_bins * 90/100

In [ ]:
minima_slit_size  =    3   # no. of bins 
minima_slit_count =  100
maxima_slit_count = 1000

advance_ratio = 0.6
cnc = 1
emergency_slope = 0
angle_look = 48

In [ ]:
min_x = no_bins*0.025
max_x = no_bins*0.975
min_y = np.min(lower_unique)*0.7
max_y = np.max(upper_unique)*1.3

In [ ]:
up_list_x_n   = x_smooth
up_list_y_n   = upper_smooth
down_list_x_n = x_smooth
down_list_y_n = lower_smooth

s_guesses            = [[], []]
s_slit_elements      = [[], []]
s_slit_sizes         = [[], []]
s_slit_slope_centers = [[], []]
s_qdrnts             = [[], []]
s_fits               = [[], []]

s_guesses_l = [[[], []], [[], []]]
s_fits_l    = [[[], []], [[], []]]

---

In [ ]:
def fig_plot():

    fig, axs = plt.subplots(dpi=600, figsize=(7,7))
    fig.suptitle("Colormap for density distribution along the radius of all voids\n"+MK+"  |  size="+str(size)+"  |  Z="+zPath+r"  |  $\sigma=$"+str(sigma_p)
                 +"cMpc/h  |  Lvl="+latex_float(lvl_ref)+r'  |  $\Delta_{CDF}=$['+str(Density_ud)+', '+str(Density_od)+']\nmoving towards '+lr[d]+'  |  step='+str(i), fontsize=10, y=0.95)
    
    
    imm = axs.hist2d(x_cut, y_cut, bins=[np.arange(0, no_bins, 1), np.arange(0, no_bins, 1)], cmin=0.1, norm=mpl.colors.LogNorm(), alpha=0.3)
    plt.scatter(x_cutt+0.5, y_cutt+0.5, c='r', s=1,   lw=0, alpha=0.6, label='Excluded cells')
    
    plt.plot(x_smooth+0.5, upper_smooth+0.5, c='blue', lw=0.5, label='Upper & Lower bounds')
    plt.plot(x_smooth+0.5, lower_smooth+0.5, c='blue', lw=0.5)
    
    
    
    divider = make_axes_locatable(axs)
    cax = divider.append_axes('right', size='5%', pad=0.05)
    cbar = plt.colorbar(imm[3], cax=cax)  # Use the 3rd element from hist2d's return (the image)
    vmin, vmax = imm[3].get_clim()
    tks, tkss = set_ticks(1, vmax)
    cbar.ax.set_yticks(tks, tkss)
    cbar.set_label("Number of voids sharing set density\nat set radius percentage", rotation=270, labelpad=15)
    
    axs.scatter([x for x in slit_elements1[0]], slit_elements1[1], c='purple', s=1, lw=0, alpha=0.6, label='slit elements')
    
    axs.plot([x for x in s_guesses_l[0][0]], s_guesses_l[0][1], c='orange', lw=0.3, label='guesses')
    axs.plot([x for x in s_guesses_l[1][0]], s_guesses_l[1][1], c='orange', lw=0.3)
    
    axs.plot([x for x in s_fits_l[0][0]], s_fits_l[0][1], c='black', lw=0.3, zorder=6, label='fits connections')
    axs.plot([x for x in s_fits_l[1][0]], s_fits_l[1][1], c='black', lw=0.3, zorder=6)
    axs.scatter([x for x in s_fits_l[0][0]], s_fits_l[0][1], c='red', lw=0, s=0.6, zorder=7, label='fits')
    axs.scatter([x for x in s_fits_l[1][0]], s_fits_l[1][1], c='red', lw=0, s=0.6, zorder=7)
    
    axs.set_xlim(-1, no_bins)
    axs.set_ylim(-1, no_bins)
    axs.set_ylabel(y_label)
    axs.set_xlabel(x_label)
    
    axs.set_yticks([10,60,110,160,200], [r"$10^{-1}$", r"$1$", r"$10^{1}$", r"$10^{2}$", f'$10^{{{round(np.log10(10**2 + (200-160)/50*(10**3-10**2)),2)}}}$'])
    axs.set_xticks(np.arange(0,201,20), np.arange(0,101,10))
    
    axs.legend(prop={'size': 6}, loc=4)
    txt1 = plots_path_0+"11___Shape_Hist/11.5___Follow_Curve_"+str(int(no_bins))+"/Z_"+zPath+"/plt/plt_d"+str(d)+'_i'+str(i)+'.png'
    plt.tight_layout()
    plt.savefig(txt1, dpi=600)
    plt.close(fig)

In [ ]:
def fig_hist():
    
    fig = plt.figure(dpi=150)

    xxs = np.linspace(min(slit_elements_rotated1[1]), max(slit_elements_rotated1[1]),2000)
    yys = [duo_norm1(x, *ff1) for x in xxs]
    
    plt.plot(xxs, yys, lw=0.1, c='green', label='Gaussian fit')
    
    plt.hist(slit_elements_rotated1[1], bins=25, alpha=0.3)
    
    
    plt.ylabel('Counts')
    plt.xlabel('The axis tangentual to the fit [units of cells]')
    txt = 'hist_d'+str(d)+'_i'+str(i)
    txt1 = plots_path_0+"11___Shape_Hist/11.5___Follow_Curve_"+str(int(no_bins))+"/Z_"+zPath+"/hist/hist_d"+str(d)+'_i'+str(i)+'.png'
    plt.title(txt)
    plt.legend()
    plt.tight_layout()
    plt.savefig(txt1, dpi=350)
    plt.close(fig)

In [ ]:
def fig_angle():
    
    fig = plt.figure(dpi=600)
    
    plt.axvline(anglees1[indx_fit_width], lw=0.3, c='green')
    
    plt.scatter(anglees, limiter, c='r', lw=0.3, label='Data from different angles')
    plt.plot(anglees, limiter, c='r', lw=0.3)
    plt.scatter(anglees, fittii, c='green', marker='x', lw=0.3, label='Fit minima')
    plt.plot(anglees1, fittii1, c='green', lw=0.3)
    
    plt.xlabel('Angle variation [degrees]')
    plt.ylabel('Counts density [counts over slit width]')
    txt = 'angle_check_d'+str(d)+'_i'+str(i)
    txt1 = plots_path_0+"11___Shape_Hist/11.5___Follow_Curve_"+str(int(no_bins))+"/Z_"+zPath+"/angle/angle_check_d"+str(d)+'_i'+str(i)+'.png'
    plt.title(txt)
    plt.legend(loc=0)
    plt.tight_layout()
    plt.savefig(txt1, dpi=350)
    plt.close(fig)

---

In [ ]:
# iterate to the right & left (left first, right later... makes no difference really)
for d in back_forward:

    starting_point1 = starting_point
    
    pmm = pm[d]
    
    i = 0
    i0 = 0   # This is in case we return to a previous value, so we can advance from the last good value where we were not backtracking.
    
    attempts = 0
    repeated_failed_attempts = 0
    keep_advancing = True
    
    while keep_advancing:
        
        # The logic behind this advancing is that we could be looking at the previous slit's width (which is at an angle) and we advance in 
        #    the x direction as half of that (angled) length and then in the y axis we follow the zeroth, first and second (on the first, 
        #    second and all to follow steps) derivatives. This gives us a good idea of what we're dealing with, but it has the issue that, 
        #    in the vertical region, we are advancing in the x direction much more than we should. What we should be doing is to advance to 
        #    the half-point of the slope, as we see below.
        
        # Check if we don't start to repeat ourselves
        if attempts >= 6:
            check_diff1 = np.sqrt((np.mean(s_fits_l[d][0][-6:-3]) - np.mean(s_fits_l[d][0][-3:]))**2 + (np.mean(s_fits_l[d][1][-6:-3]) - np.mean(s_fits_l[d][1][-3:]))**2)
            check_diff2 = np.mean(s_slit_sizes[d][-6:])*advance_ratio
        
        
        if i0 != 0:
            if advance_ratio + 0.2 + 0.1*i0 < 1.3: advance_ratio1 = advance_ratio + 0.2 + 0.1*i0
            else:                                  advance_ratio1 = 1.3
        
        elif ((attempts >= 6) and (check_diff1 < check_diff2)):
            if advance_ratio + 0.2 + 0.1*i0 < 1.3: advance_ratio1 = advance_ratio + 0.2 + 0.1*i0
            else:                                  advance_ratio1 = 1.3
        
        else:                                      advance_ratio1 = advance_ratio
        
        
        
        if i != 0:
            travel_length = s_slit_sizes[d][-1] * advance_ratio1
            if (3 <= i < 6) or ((attempts >= 6) and (check_diff1 > check_diff2)):
                x1 = (s_fits[d][-1][0]-s_fits[d][-2][0])
                x0 = (s_fits[d][-2][0]-s_fits[d][-3][0])
                if x1+x0 == 0:
                    ratio = 1
                else:
                    ratio = (x1*s_slit_slope_centers[d][-1] + x0*s_slit_slope_centers[d][-2])/((x1+x0)/2)
                starting_point_x = s_fits[d][-1][0] - pmm*(travel_length / np.sqrt(1/ratio**2 + 1))
                starting_point_y = s_fits[d][-1][1] + pmm*(travel_length / np.sqrt(ratio**2 + 1))
                starting_point1 = [starting_point_x, starting_point_y]
            else:
                ratio = s_slit_slope_centers[d][-1]
                if emergency_slope == 1:
                    ratio = 10**6
                starting_point_x = s_fits[d][-1][0] - pmm*(travel_length / np.sqrt(1/ratio**2 + 1))
                starting_point_y = s_fits[d][-1][1] + pmm*(travel_length / np.sqrt(ratio**2 + 1))
                starting_point1 = [starting_point_x, starting_point_y]
            
            
        s_guesses[  d].append(   starting_point1   )
        s_guesses_l[d][0].append(starting_point1[0])
        s_guesses_l[d][1].append(starting_point1[1])
        
        
            
        
        slope_center, qdrnt = slope_finder(starting_point1, up_list_x_n, up_list_y_n, down_list_x_n, down_list_y_n)
        slope_centers = [[slope_center], [slope_center]]
        
        
        gt = diff_finder(slope_center, qdrnt)
        sp_rotated = rotator_3000([[starting_point1[0]], [starting_point1[1]]], gt, qdrnt)
        starting_point_rotated = [sp_rotated[0][0], sp_rotated[1][0]]
        
        
        xy_rotated = rotator_3000([x_cut, y_cut], gt, qdrnt)
        xy_rotated_ordered = [[], []]
        order_index_xy = sorted(range(len(xy_rotated[0])), key=lambda k: xy_rotated[0][k])
        for iii in range(len(xy_rotated[0])):
            xy_rotated_ordered[0].append(xy_rotated[0][order_index_xy[iii]])
            xy_rotated_ordered[1].append(xy_rotated[1][order_index_xy[iii]])
        
        
        if i == 0: slit_elements_rotated, slit_size = elements_finder(xy_rotated_ordered[0], xy_rotated_ordered[1], starting_point_rotated, minima_slit_size, minima_slit_count, maxima_slit_count, 0,                   length_bin_0)
        else:      slit_elements_rotated, slit_size = elements_finder(xy_rotated_ordered[0], xy_rotated_ordered[1], starting_point_rotated, minima_slit_size, minima_slit_count, maxima_slit_count, s_slit_sizes[d][-1], length_bin_0)
        
        
        slit_mean_rotated, slr, ff, slit_fit_width = mean_by_fitting2(slit_elements_rotated, cnc)
        add = rotator_3000([[slr], [slit_mean_rotated]], -gt, qdrnt)
        slit_mean = [add[0][0], add[1][0]]
        
        
        
        slit_size_list = []
        slit_count_list = []
        slit_fit_width_list = []


        
        if   qdrnt in [1,4]: tan_slope_center = math.atan(slope_center)
        elif qdrnt in [2,3]: tan_slope_center = np.pi + math.atan(slope_center)
        
        
        
        for j in range(int(angle_look+1+1)):
            
            if j != angle_look+1: tann = tan_slope_center + (-angle_look+2*j)       /180*np.pi
            else:                 tann = tan_slope_center + anglees1[indx_fit_width]/180*np.pi
            
            qdrnt1 = get_quadrant(tann)
            slope_center1 = math.tan(tann)
            
            
            gt1 = diff_finder(slope_center1, qdrnt1)
            sp_rotated1 = rotator_3000([[slit_mean[0]], [slit_mean[1]]], gt1, qdrnt1)
            starting_point_rotated1 = [sp_rotated1[0][0], sp_rotated1[1][0]]
            
            
            xy_rotated1 = rotator_3000([x_cut, y_cut], gt1, qdrnt1)
            xy_rotated_ordered1 = [[], []]
            order_index_xy1 = sorted(range(len(xy_rotated1[0])), key=lambda k: xy_rotated1[0][k])
            for iii in range(len(xy_rotated1[0])):
                xy_rotated_ordered1[0].append(xy_rotated1[0][order_index_xy1[iii]])
                xy_rotated_ordered1[1].append(xy_rotated1[1][order_index_xy1[iii]])
    
    
            slit_elements_rotated1, slit_size1 = elements_finder(xy_rotated_ordered1[0], xy_rotated_ordered1[1], starting_point_rotated1, minima_slit_size, minima_slit_count, maxima_slit_count, slit_size, length_bin_0)
            # save
            slit_size_list.append(slit_size1)
            slit_count_list.append(len(slit_elements_rotated1[0]))
    
    
            
            if j == angle_look:
                
                anglees = [-angle_look+2*ii                       for ii in range(int(angle_look+1))]
                limiter = [slit_count_list[ii]/slit_size_list[ii] for ii in range(int(angle_look+1))]
                
                pzro = [0, 5, min(limiter)]
                fitted_paras, s = scipy.optimize.curve_fit(angling_fit, anglees, limiter, p0=pzro, maxfev=1000000)
                
                fittii  = [angling_fit(x, *fitted_paras) for x in anglees ]
                anglees1 = np.linspace(anglees[0]*1.3, anglees[-1]*1.3,1000)
                fittii1 = [angling_fit(x, *fitted_paras) for x in anglees1]
                
                indx_fit_width = np.argmin(fittii1)
                fig_angle()
        
        
        ####################################################################################################
        
        
        slit_elements1 = rotator_3000(slit_elements_rotated1, -gt1, qdrnt1)
        slit_mean_rotated1, slr1, ff1, slit_fit_width1 = mean_by_fitting2(slit_elements_rotated1, cnc)
        add1 = rotator_3000([[slr1], [slit_mean_rotated1]], -gt1, qdrnt1)
        slit_mean1 = [add1[0][0], add1[1][0]]
        
        
        if slit_mean1 in s_fits[d][-10:-1]:
            i0 += 1
            #attempts = 0
        
        
        else:
            i0 = 0
            if (attempts >= 6) and (check_diff1 < check_diff2): attempts  = 3
            else:                                               attempts += 1
            
            s_slit_elements[     d].append(slit_elements1)
            s_slit_sizes[        d].append(slit_size1)
            s_slit_slope_centers[d].append(slope_center1)
            s_qdrnts[            d].append(qdrnt1)
            
            s_fits[  d].append(   slit_mean1)
            s_fits_l[d][0].append(slit_mean1[0])
            s_fits_l[d][1].append(slit_mean1[1])
            
            fig_hist()
            fig_plot()
            
            i += 1
        


        # Check if we do, in fact, advance in the direction of this run.
        check_advance = False
        if i != 1:
            if d == 0: check_advance = s_fits_l[d][0][-1] >= np.min(s_fits_l[d][0][:-1])
            else:      check_advance = s_fits_l[d][0][-1] <= np.max(s_fits_l[d][0][:-1])

        # If we don't, we try again using the above mechanisms to extend our range and all that.
        # But if we tried 4 times, time to give up.
        # But if it works eventually, we reset the counting... maybe it was a rough spot.
        if check_advance:
            repeated_failed_attempts += 1
            if repeated_failed_attempts == 5:
                keep_advancing = False
        else:
            repeated_failed_attempts = 0
        
        # Break the while loop when we get outside the input range.
        if (slit_mean1[0] < min_x) or (slit_mean1[0] > max_x) or (slit_mean1[1] < min_y) or (slit_mean1[1] > max_y):
            keep_advancing = False

---

In [ ]:
the_list_of_lists =       [ s_guesses,      s_slit_elements,      s_slit_sizes,      s_slit_slope_centers,      s_qdrnts,      s_fits,      s_guesses_l,      s_fits_l    ]
the_list_of_lists_names = ['s_guesses.pk', 's_slit_elements.pk', 's_slit_sizes.pk', 's_slit_slope_centers.pk', 's_qdrnts.pk', 's_fits.pk', 's_guesses_l.pk', 's_fits_l.pk']

for i in range(len(the_list_of_lists)):
    with open(plots_path_0+"11___Shape_Hist/11.5___Follow_Curve_"+str(int(no_bins))+"/Z_"+zPath+"/"+the_list_of_lists_names[i], 'wb') as f:
        pkl.dump(the_list_of_lists[i], f)

---
---
---